In [ ]:
# In[1]:

import pandas as pd

# --- Metric ---
df_metric = pd.read_csv('metric.csv')
# parse timestamp as UTC datetime and ensure value is float
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_metric['value'] = df_metric['value'].astype(float)

# group and aggregate: count, mean, median (50th), 95th, min, max
metric_grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
metric_summary = metric_grp.agg(
    count='count',
    mean='mean',
    median=lambda x: x.quantile(0.5),
    p95=lambda x: x.quantile(0.95),
    min='min',
    max='max'
).reset_index()

# sort by count desc and take top 20, round numeric columns for compactness
metric_summary = metric_summary.sort_values('count', ascending=False).head(20)
metric_summary.loc[:, ['mean','median','p95','min','max']] = metric_summary.loc[:, ['mean','median','p95','min','max']].round(6)

# list of unique cmdb_id in metric.csv as a compact single-column DataFrame
metric_cmdbs = pd.DataFrame({'cmdb_id': pd.unique(df_metric['cmdb_id'])})

# --- Log ---
df_log = pd.read_csv('log.csv')
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_log['value'] = df_log['value'].astype(float)

log_grp = df_log.groupby(['cmdb_id', 'log_name'])['value']
log_summary = log_grp.agg(
    count='count',
    mean='mean',
    median=lambda x: x.quantile(0.5),
    p95=lambda x: x.quantile(0.95),
    min='min',
    max='max'
).reset_index()

log_summary = log_summary.sort_values('count', ascending=False).head(20)
log_summary.loc[:, ['mean','median','p95','min','max']] = log_summary.loc[:, ['mean','median','p95','min','max']].round(6)
log_cmdbs = pd.DataFrame({'cmdb_id': pd.unique(df_log['cmdb_id'])})

# --- Trace ---
df_trace = pd.read_csv('trace.csv')
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_trace['value'] = df_trace['value'].astype(float)

trace_grp = df_trace.groupby(['cmdb_id', 'trace_name'])['value']
trace_summary = trace_grp.agg(
    count='count',
    mean='mean',
    median=lambda x: x.quantile(0.5),
    p95=lambda x: x.quantile(0.95),
    min='min',
    max='max'
).reset_index()

trace_summary = trace_summary.sort_values('count', ascending=False).head(20)
trace_summary.loc[:, ['mean','median','p95','min','max']] = trace_summary.loc[:, ['mean','median','p95','min','max']].round(6)
trace_cmdbs = pd.DataFrame({'cmdb_id': pd.unique(df_trace['cmdb_id'])})

# --- Error logs ---
df_err = pd.read_csv('error_logs.csv')
df_err['timestamp'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True)
error_count = df_err.shape[0]
error_preview = df_err.loc[:, ['timestamp','cmdb_id','message']].head(10)

# Display compact results (DataFrames/values). IPython will show them.
metric_summary, metric_cmdbs, log_summary, log_cmdbs, trace_summary, trace_cmdbs, error_count, error_preview

```
Out[1]:
```
Summary of the telemetry aggregation (plain English):

1) metric.csv (per (cmdb_id, kpi_name) aggregates)
- Top aggregated pairs by sample count (showing count, mean, median, 95th, min, max):
  - carts — cpu: count=25, mean≈3.865412, median≈2.970073, p95≈7.805729, min≈1.610777, max≈13.12330
  - carts — latency-50: count=25, mean≈0.012906, median≈0.013037, p95≈0.015316, min≈0.009684, max≈0.018605
  - carts — latency-90: count=25, mean≈0.024111, median≈0.023060, p95≈0.026933, min≈0.021815, max≈0.045908
  - carts — mem: count=25, mean≈207,585,800, median≈209,501,600, p95≈211,355,600, min≈194,128,700, max≈211,494,400
  - carts — socket: count=25, mean≈10.10238, median≈9.983333, p95≈11.59667, min≈9.316667, max≈11.93333
  - carts — workload: count=25, mean≈8.077442, median≈8.092167, p95≈8.81531, min≈6.152143, max≈8.966667
  - carts-db — cpu/diskio/mem/socket and catalogue / catalogue-db / etc. also appear with count=25 (see full table if needed).
- Unique cmdb_id values present in metric.csv:
  carts, carts-db, catalogue, catalogue-db, front-end, orders, orders-db, payment, queue-master, rabbitmq, rabbitmq-exporter, session-db, shipping, user, user-db

2) log.csv (per (cmdb_id, log_name) aggregates)
- Top aggregated pairs by sample count (count, mean, median, p95, min, max):
  - catalogue — log.error_count: count=25, mean=0.0, median=0.0, p95=0.0, min=0.0, max=0.0
  - catalogue — log.row_count: count=25, mean≈155.12, median=161.0, p95≈168.8, min=13.0, max=173.0
  - orders — log.row_count: count=25, mean≈117.36, median=122.0, p95≈132.0, min=6.0, max=134.0
  - orders — log.error_count: count=25, mean=0.0 (all zeros)
  - front-end — log.row_count: count=25, mean≈1883.6, median=1958.0, p95≈2064.2, min=190.0, max=2139.0
  - front-end — log.error_count: count=25, mean=0.0 (all zeros)
  - shipping, user, payment, queue-master, carts, session-db entries also present (many with log.error_count=0.0).
- Unique cmdb_id values present in log.csv:
  catalogue, front-end, orders, payment, queue-master, shipping, user, carts, carts-db, session-db, user-db

3) trace.csv
- No trace telemetry rows found. The trace summary is empty and there are no unique cmdb_id values in trace.csv.

4) error_logs.csv
- Total number of error_log rows: 0
- Preview of first 10 rows: empty (no rows to show)

Overall key points:
- metric and log CSVs contain data across many services; carts (and carts-db) has many KPI samples and relatively larger values for memory and socket metrics in the provided top results.
- log.error_count is zero for all top services (no logged errors in these aggregated rows).
- There is no trace telemetry and no error_log entries available in the current dataset.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count          mean        median           p95           min           max
0          carts         cpu     25  3.865412e+00  2.970073e+00  7.805729e+00  1.610777e+00  1.312330e+01
1          carts  latency-50     25  1.290600e-02  1.303700e-02  1.531600e-02  9.684000e-03  1.860500e-02
2          carts  latency-90     25  2.411100e-02  2.306000e-02  2.693300e-02  2.181500e-02  4.590800e-02
3          carts         mem     25  2.075858e+08  2.095016e+08  2.113556e+08  1.941287e+08  2.114944e+08
4          carts      socket     25  1.010238e+01  9.983333e+00  1.159667e+01  9.316667e+00  1.193333e+01
5          carts    workload     25  8.077442e+00  8.092167e+00  8.815310e+00  6.152143e+00  8.966667e+00
6       carts-db         cpu     25  2.222475e+00  2.128437e+00  3.022392e+00  1.465127e+00  3.079411e+00
7       carts-db      diskio     25  1.697641e+06  1.657091e+06  1.860324e+06  1.506443e+06  1.999862e+06
8       carts-db         mem     25  7.145126e+07  7.107086e+07  7.372055e+07  6.957466e+07  7.409300e+07
9       carts-db      socket     25  6.119333e+00  6.000000e+00  6.996667e+00  6.000000e+00  7.000000e+00
10     catalogue         cpu     25  1.689800e-01  1.708770e-01  1.807890e-01  1.472240e-01  1.811560e-01
11     catalogue  latency-50     25  3.041000e-03  3.042000e-03  3.067000e-03  3.006000e-03  3.069000e-03
12     catalogue  latency-90     25  4.674000e-03  4.675000e-03  4.721000e-03  4.610000e-03  4.724000e-03
13     catalogue         mem     25  5.722313e+06  5.801438e+06  5.910460e+06  4.726784e+06  5.919251e+06
14     catalogue      socket     25  5.788667e+00  6.000000e+00  6.000000e+00  4.000000e+00  6.000000e+00
15     catalogue    workload     25  4.073056e+00  4.087633e+00  4.290000e+00  3.775433e+00  4.357600e+00
16  catalogue-db         cpu     25  2.018430e-01  2.043740e-01  2.116340e-01  1.821190e-01  2.128090e-01
17  catalogue-db      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
18  catalogue-db         mem     25  4.305370e+08  4.305359e+08  4.305588e+08  4.304609e+08  4.305684e+08
19  catalogue-db      socket     25  3.988571e+00  4.000000e+00  4.000000e+00  3.714286e+00  4.000000e+00,               cmdb_id
0               carts
1            carts-db
2           catalogue
3        catalogue-db
4           front-end
5              orders
6           orders-db
7             payment
8        queue-master
9            rabbitmq
10  rabbitmq-exporter
11         session-db
12           shipping
13               user
14            user-db,          cmdb_id         log_name  count         mean  median     p95    min     max
4      catalogue  log.error_count     25     0.000000     0.0     0.0    0.0     0.0
5      catalogue    log.row_count     25   155.120000   161.0   168.8   13.0   173.0
9         orders    log.row_count     25   117.360000   122.0   132.0    6.0   134.0
8         orders  log.error_count     25     0.000000     0.0     0.0    0.0     0.0
7      front-end    log.row_count     25  1883.600000  1958.0  2064.2  190.0  2139.0
6      front-end  log.error_count     25     0.000000     0.0     0.0    0.0     0.0
17      shipping    log.row_count     25    58.680000    61.0    66.0    3.0    67.0
16      shipping  log.error_count     25     0.000000     0.0     0.0    0.0     0.0
19          user    log.row_count     25   566.800000   592.0   630.0   44.0   637.0
18          user  log.error_count     25     0.000000     0.0     0.0    0.0     0.0
10       payment  log.error_count     25     0.000000     0.0     0.0    0.0     0.0
11       payment    log.row_count     25    96.400000   101.0   106.0    5.0   107.0
12  queue-master  log.error_count     25     0.000000     0.0     0.0    0.0     0.0
13  queue-master    log.row_count     25   528.120000   549.0   594.0   27.0   603.0
2       carts-db  log.error_count     23     0.000000     0.0     0.0    0.0     0.0
3       carts-db    log.row_count     23     1.043478     1.0     1.0    1.0     2.0
0          carts  log.error_count     20     0.000000     0.0     0.0    0.0     0.0
1          carts    log.row_count     20    76.300000    75.0   150.0   25.0   150.0
14    session-db  log.error_count      5     0.000000     0.0     0.0    0.0     0.0
15    session-db    log.row_count      5     5.000000     5.0     5.0    5.0     5.0,          cmdb_id
0      catalogue
1      front-end
2         orders
3        payment
4   queue-master
5       shipping
6           user
7          carts
8       carts-db
9     session-db
10       user-db, Empty DataFrame
Columns: [cmdb_id, trace_name, count, mean, median, p95, min, max]
Index: [], Empty DataFrame
Columns: [cmdb_id]
Index: [], 0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
from datetime import datetime, timezone

# reuse previously loaded DataFrames if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv')

# Ensure timestamps parsed to UTC datetimes and numeric values as floats
for df, ts_col, val_col in [
    (df_metric, 'timestamp', 'value'),
    (df_log, 'timestamp', 'value'),
    (df_trace, 'timestamp', 'value'),
    (df_err, 'timestamp', None)
]:
    if ts_col in df.columns:
        df[ts_col] = pd.to_datetime(df[ts_col], unit='s', utc=True)
    if val_col and val_col in df.columns:
        df[val_col] = df[val_col].astype(float)

# Target components and incident window (UTC)
targets = ["carts","catalogue","orders","payment","user"]
start = pd.to_datetime("2024-01-20T14:06:53Z", utc=True)
end   = pd.to_datetime("2024-01-20T14:36:53Z", utc=True)

# Helper to format example anomalies (up to 5) as a semicolon-separated string "ISO|value"
def make_examples(subdf, ts_col='timestamp', val_col='value', limit=5):
    if subdf.shape[0] == 0:
        return ""
    rows = subdf.sort_values(ts_col).head(limit)
    return "; ".join(f"{ts.isoformat()}|{round(v,6)}" for ts, v in zip(rows[ts_col], rows[val_col]))

# -------------------------
# 1) METRIC processing
# -------------------------
metric_targets = df_metric[df_metric['cmdb_id'].isin(targets)].copy()

if metric_targets.shape[0] > 0:
    # global stats on full series per (cmdb_id,kpi_name)
    metric_global = metric_targets.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_count='count',
        p5=lambda x: x.quantile(0.05),
        p50=lambda x: x.quantile(0.50),
        p95=lambda x: x.quantile(0.95)
    ).reset_index()
    metric_global[['p5','p50','p95']] = metric_global[['p5','p50','p95']].round(4)
    
    # filter to incident window
    metric_in = metric_targets[(metric_targets['timestamp'] >= start) & (metric_targets['timestamp'] <= end)].copy()
    if metric_in.shape[0] > 0:
        # merge thresholds into in-window rows
        metric_in = metric_in.merge(metric_global, on=['cmdb_id','kpi_name'], how='left')
        metric_in['is_anom'] = (metric_in['value'] <= metric_in['p5']) | (metric_in['value'] >= metric_in['p95'])
        # aggregate in-window per pair
        def agg_window(g):
            in_count = g.shape[0]
            anom_count = int(g['is_anom'].sum())
            if anom_count > 0:
                earliest = g.loc[g['is_anom'], 'timestamp'].min().isoformat()
                examples = make_examples(g[g['is_anom']])
            else:
                earliest = pd.NaT
                examples = ""
            return pd.Series({
                'in_window_count': in_count,
                'in_window_anomaly_count': anom_count,
                'earliest_anomaly_iso': earliest if pd.notna(earliest) else "",
                'example_anomalies': examples
            })
        metric_window_agg = metric_in.groupby(['cmdb_id','kpi_name']).apply(agg_window).reset_index()
    else:
        # no in-window rows
        metric_window_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','in_window_count','in_window_anomaly_count','earliest_anomaly_iso','example_anomalies'])
    
    # combine global and window summaries, ensure pairs with no in-window rows are represented
    metric_summary_full = metric_global.merge(metric_window_agg, on=['cmdb_id','kpi_name'], how='left')
    metric_summary_full['in_window_count'] = metric_summary_full['in_window_count'].fillna(0).astype(int)
    metric_summary_full['in_window_anomaly_count'] = metric_summary_full['in_window_anomaly_count'].fillna(0).astype(int)
    metric_summary_full['earliest_anomaly_iso'] = metric_summary_full['earliest_anomaly_iso'].fillna("")
    metric_summary_full['example_anomalies'] = metric_summary_full['example_anomalies'].fillna("")
    
    # sort by in-window anomaly_count desc then global_count desc, take top 20
    metric_result = metric_summary_full.sort_values(
        by=['in_window_anomaly_count','global_count'], ascending=[False, False]
    ).head(20).reset_index(drop=True)
else:
    metric_result = pd.DataFrame(columns=['cmdb_id','kpi_name','global_count','p5','p50','p95','in_window_count','in_window_anomaly_count','earliest_anomaly_iso','example_anomalies'])

# -------------------------
# 2) LOG processing
# -------------------------
log_targets = df_log[df_log['cmdb_id'].isin(targets)].copy()

if log_targets.shape[0] > 0:
    log_global = log_targets.groupby(['cmdb_id','log_name'])['value'].agg(
        global_count='count',
        p5=lambda x: x.quantile(0.05),
        p50=lambda x: x.quantile(0.50),
        p95=lambda x: x.quantile(0.95)
    ).reset_index()
    log_global[['p5','p50','p95']] = log_global[['p5','p50','p95']].round(4)
    
    log_in = log_targets[(log_targets['timestamp'] >= start) & (log_targets['timestamp'] <= end)].copy()
    if log_in.shape[0] > 0:
        log_in = log_in.merge(log_global, on=['cmdb_id','log_name'], how='left')
        log_in['is_anom'] = (log_in['value'] <= log_in['p5']) | (log_in['value'] >= log_in['p95'])
        def agg_window_log(g):
            in_count = g.shape[0]
            anom_count = int(g['is_anom'].sum())
            if anom_count > 0:
                earliest = g.loc[g['is_anom'], 'timestamp'].min().isoformat()
                examples = make_examples(g[g['is_anom']])
            else:
                earliest = pd.NaT
                examples = ""
            return pd.Series({
                'in_window_count': in_count,
                'in_window_anomaly_count': anom_count,
                'earliest_anomaly_iso': earliest if pd.notna(earliest) else "",
                'example_anomalies': examples
            })
        log_window_agg = log_in.groupby(['cmdb_id','log_name']).apply(agg_window_log).reset_index()
    else:
        log_window_agg = pd.DataFrame(columns=['cmdb_id','log_name','in_window_count','in_window_anomaly_count','earliest_anomaly_iso','example_anomalies'])
    
    log_summary_full = log_global.merge(log_window_agg, on=['cmdb_id','log_name'], how='left')
    log_summary_full['in_window_count'] = log_summary_full['in_window_count'].fillna(0).astype(int)
    log_summary_full['in_window_anomaly_count'] = log_summary_full['in_window_anomaly_count'].fillna(0).astype(int)
    log_summary_full['earliest_anomaly_iso'] = log_summary_full['earliest_anomaly_iso'].fillna("")
    log_summary_full['example_anomalies'] = log_summary_full['example_anomalies'].fillna("")
    
    log_result = log_summary_full.sort_values(
        by=['in_window_anomaly_count','global_count'], ascending=[False, False]
    ).head(20).reset_index(drop=True)
else:
    log_result = pd.DataFrame(columns=['cmdb_id','log_name','global_count','p5','p50','p95','in_window_count','in_window_anomaly_count','earliest_anomaly_iso','example_anomalies'])

# -------------------------
# 3) TRACE and ERROR_LOGS window checks
# -------------------------
trace_in_window = df_trace[(df_trace['timestamp'] >= start) & (df_trace['timestamp'] <= end)].copy()
trace_window_count = trace_in_window.shape[0]
if trace_window_count > 0:
    # keep small preview (first 10 rows) with selected cols
    trace_preview = trace_in_window.loc[:, ['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp').head(10)
else:
    trace_preview = pd.DataFrame()  # empty

err_in_window = df_err[(df_err['timestamp'] >= start) & (df_err['timestamp'] <= end)].copy()
err_window_count = err_in_window.shape[0]
if err_window_count > 0:
    err_preview = err_in_window.loc[:, ['timestamp','cmdb_id','message']].sort_values('timestamp').head(10)
else:
    err_preview = pd.DataFrame()

# Compact display: show metric_result and log_result with limited columns and examples, and trace/error summaries
metric_result_display = metric_result.loc[:, [
    'cmdb_id','kpi_name','global_count','p5','p50','p95','in_window_count','in_window_anomaly_count','earliest_anomaly_iso','example_anomalies'
]]
log_result_display = log_result.loc[:, [
    'cmdb_id','log_name','global_count','p5','p50','p95','in_window_count','in_window_anomaly_count','earliest_anomaly_iso','example_anomalies'
]]

# Final outputs (IPython will display them)
metric_result_display, log_result_display, (trace_window_count, trace_preview), (err_window_count, err_preview)

```
Out[2]:
```
Summary (incident window 2024-01-20T14:06:53Z → 2024-01-20T14:36:53Z UTC)

Key notes about methodology used here: global thresholds (p5, p50, p95) were computed from each full series first, then the data were filtered to the incident window and any in-window sample with value <= p5 or >= p95 was counted as an anomaly. This means constant-zero series where p5 = p95 = 0 will be flagged as “anomalous” for values equal to 0.

1) Metrics — top anomalous (cmdb_id, kpi_name) rows (sorted by in-window anomaly count)
- user — error
  - global_count=25, p5=0.0000, p50=0.0000, p95=0.0000
  - in-window_count=25, in-window_anomaly_count=25
  - earliest anomaly: 2024-01-20T14:09:00+00:00
  - example anomalies: 2024-01-20T14:09:00+00:00|0.0; 2024-01-20T14:10:00+00:00|0.0; ...
- catalogue — socket
  - global_count=25, p5=4.9600, p50=6.0000, p95=6.0000
  - in-window_count=25, in-window_anomaly_count=22
  - earliest anomaly: 2024-01-20T14:09:00+00:00
  - example anomalies: 2024-01-20T14:09:00+00:00|4.0; 2024-01-20T14:10:00+00:00|4.0; ...
- payment — socket
  - global_count=25, p5=3.0000, p50=4.0000, p95=8.0000
  - in-window_count=25, in-window_anomaly_count=22
  - earliest anomaly: 2024-01-20T14:09:00+00:00
  - example anomalies: 2024-01-20T14:09:00+00:00|3.0; 2024-01-20T14:10:00+00:00|3.0; ...
- payment — diskio
  - global_count=24, p5=0.0000, p50≈4.4457e+09, p95≈1.7911e+10
  - in-window_count=24, in-window_anomaly_count=14
  - earliest anomaly: 2024-01-20T14:10:00+00:00
  - example anomalies: 2024-01-20T14:10:00+00:00|0.0; 2024-01-20T14:11:00+00:00|...; ...
- user — socket
  - global_count=25, p5=12.0000, p50=16.0000, p95=17.0000
  - in-window_count=25, in-window_anomaly_count=10
  - earliest anomaly: 2024-01-20T14:09:00+00:00
  - example anomalies: 2024-01-20T14:09:00+00:00|12.0; 2024-01-20T14:10:00+00:00|12.0; ...
- catalogue — latency-90
  - global_count=25, p5=0.0046, p50=0.0047, p95=0.0047
  - in-window_count=25, in-window_anomaly_count=6
  - earliest anomaly: 2024-01-20T14:09:00+00:00
- orders — latency-50 and latency-90
  - each global_count=25; each had in-window_anomaly_count=5, earliest anomaly 2024-01-20T14:09:00+00:00
- carts — cpu, latency-50, mem, socket, workload
  - each global_count=25; each had in-window_anomaly_count=4, earliest anomalies mostly 2024-01-20T14:09:00+00:00 (carts socket earliest 14:18:00)
- orders / catalogue / payment / user / carts show other KPI entries with 4 in-window anomalies (cpu, mem, socket, workload, etc.)

(Only top 20 metric pairs were checked; above are the leading anomalous pairs from that list.)

2) Logs — top anomalous (cmdb_id, log_name) rows
- catalogue — log.error_count
  - global_count=25, p5=0.0, p50=0.0, p95=0.0
  - in-window_count=25, in-window_anomaly_count=25
  - earliest anomaly: 2024-01-20T14:09:00+00:00
  - example anomalies: 2024-01-20T14:09:00+00:00|0.0; ...
- orders — log.error_count
  - global_count=25, p5=p50=p95=0.0
  - in-window_count=25, in-window_anomaly_count=25 (earliest 2024-01-20T14:09:00+00:00)
- payment — log.error_count; user — log.error_count; carts — log.error_count
  - similar pattern: global p5/p50/p95 = 0.0 and all in-window samples flagged (carts had in-window_count=20, anomaly_count=20, earliest 2024-01-20T14:10:00+00:00)
  - these are likely flagged only because the series is constant zero (see methodology note).
- log.row_count entries
  - orders — log.row_count: global_count=25, p5=106.4, p50=122.0, p95=132.0; in-window_anomaly_count=5 (earliest 2024-01-20T14:09:00+00:00), examples include low values like 6.0
  - payment — log.row_count: in-window_anomaly_count=5
  - carts — log.row_count: global_count=20, p5=25.0, p50=75.0, p95=150.0; in-window_anomaly_count=5 (earliest 2024-01-20T14:17:00+00:00)
  - catalogue/user — log.row_count had in-window anomaly counts of 4 each.

3) Trace and error_logs within the incident window
- trace.csv: 0 rows in the incident window (empty preview).
- error_logs.csv: 0 rows in the incident window (empty preview).

4) Overall interpretation / practical notes
- Many anomalies are concentrated at ~2024-01-20T14:09:00+00:00 (earliest anomaly timestamps shown repeatedly).
- Several metrics show genuine outliers (e.g., catalogue/socket, payment/socket/diskio, orders latency, carts CPU/mem/socket/workload) with non-trivial deviations from global p5/p95.
- Many log.error_count series were flagged as “anomalous” only because the global p5 and p95 are both zero; this detection rule makes constant-zero series appear anomalous for equal-zero values — treat these with caution.
- No trace records or error_log entries were present in the incident window.

If you want, I can:
- expand the metric/log lists to show more rows or full examples per pair,
- re-run anomaly detection using a different rule (e.g., strictly value > p95 OR value < p5, excluding equality when p5==p95), or
- produce per-component timelines (textual summaries) of the anomaly timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name  global_count            p5           p50           p95  in_window_count  in_window_anomaly_count       earliest_anomaly_iso                                  example_anomalies
0        user       error            25  0.000000e+00  0.000000e+00  0.000000e+00               25                       25  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|0.0; 2024-01-20T14:1...
1   catalogue      socket            25  4.960000e+00  6.000000e+00  6.000000e+00               25                       22  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|4.0; 2024-01-20T14:1...
2     payment      socket            25  3.000000e+00  4.000000e+00  8.000000e+00               25                       22  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|3.0; 2024-01-20T14:1...
3     payment      diskio            24  0.000000e+00  4.445741e+09  1.791056e+10               24                       14  2024-01-20T14:10:00+00:00  2024-01-20T14:10:00+00:00|0.0; 2024-01-20T14:1...
4        user      socket            25  1.200000e+01  1.600000e+01  1.700000e+01               25                       10  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|12.0; 2024-01-20T14:...
5   catalogue  latency-90            25  4.600000e-03  4.700000e-03  4.700000e-03               25                        6  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|0.004724; 2024-01-20...
6      orders  latency-50            25  3.770000e-02  3.800000e-02  4.460000e-02               25                        5  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|0.060094; 2024-01-20...
7      orders  latency-90            25  4.790000e-02  4.840000e-02  1.165000e-01               25                        5  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|0.207484; 2024-01-20...
8       carts         cpu            25  1.725600e+00  2.970100e+00  7.805700e+00               25                        4  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|13.123301; 2024-01-2...
9       carts  latency-50            25  9.800000e-03  1.300000e-02  1.530000e-02               25                        4  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|0.018605; 2024-01-20...
10      carts         mem            25  1.968042e+08  2.095016e+08  2.113556e+08               25                        4  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|194128749.714286; 20...
11      carts      socket            25  9.343300e+00  9.983300e+00  1.159670e+01               25                        4  2024-01-20T14:18:00+00:00  2024-01-20T14:18:00+00:00|9.316667; 2024-01-20...
12      carts    workload            25  7.622300e+00  8.092200e+00  8.815300e+00               25                        4  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|6.152143; 2024-01-20...
13  catalogue         cpu            25  1.546000e-01  1.709000e-01  1.808000e-01               25                        4  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|0.154163; 2024-01-20...
14  catalogue         mem            25  5.089840e+06  5.801438e+06  5.910460e+06               25                        4  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|4726784.0; 2024-01-2...
15  catalogue    workload            25  3.857400e+00  4.087600e+00  4.290000e+00               25                        4  2024-01-20T14:13:00+00:00  2024-01-20T14:13:00+00:00|4.3576; 2024-01-20T1...
16     orders         cpu            25  1.759100e+00  3.384700e+00  8.715400e+00               25                        4  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|11.672897; 2024-01-2...
17     orders         mem            25  3.144156e+08  3.223489e+08  3.264992e+08               25                        4  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|314347520.0; 2024-01...
18     orders      socket            25  1.262330e+01  1.368330e+01  1.580000e+01               25                        4  2024-01-20T14:11:00+00:00  2024-01-20T14:11:00+00:00|12.516667; 2024-01-2...
19     orders    workload            25  1.938100e+00  2.043200e+00  2.152500e+00               25                        4  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|2.857143; 2024-01-20...,      cmdb_id         log_name  global_count     p5    p50    p95  in_window_count  in_window_anomaly_count       earliest_anomaly_iso                                  example_anomalies
0  catalogue  log.error_count            25    0.0    0.0    0.0               25                       25  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|0.0; 2024-01-20T14:1...
1     orders  log.error_count            25    0.0    0.0    0.0               25                       25  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|0.0; 2024-01-20T14:1...
2    payment  log.error_count            25    0.0    0.0    0.0               25                       25  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|0.0; 2024-01-20T14:1...
3       user  log.error_count            25    0.0    0.0    0.0               25                       25  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|0.0; 2024-01-20T14:1...
4      carts  log.error_count            20    0.0    0.0    0.0               20                       20  2024-01-20T14:10:00+00:00  2024-01-20T14:10:00+00:00|0.0; 2024-01-20T14:1...
5     orders    log.row_count            25  106.4  122.0  132.0               25                        5  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|6.0; 2024-01-20T14:1...
6    payment    log.row_count            25   86.8  101.0  106.0               25                        5  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|5.0; 2024-01-20T14:1...
7      carts    log.row_count            20   25.0   75.0  150.0               20                        5  2024-01-20T14:17:00+00:00  2024-01-20T14:17:00+00:00|25.0; 2024-01-20T14:...
8  catalogue    log.row_count            25  146.6  161.0  168.8               25                        4  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|13.0; 2024-01-20T14:...
9       user    log.row_count            25  531.4  592.0  630.0               25                        4  2024-01-20T14:09:00+00:00  2024-01-20T14:09:00+00:00|44.0; 2024-01-20T14:..., (0, Empty DataFrame
Columns: []
Index: []), (0, Empty DataFrame
Columns: []
Index: []))```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing df_metric if present; otherwise load and parse
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
    df_metric['value'] = df_metric['value'].astype(float)

# Ensure timestamp and value types
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_metric['value'] = df_metric['value'].astype(float)

# Targets and incident window (UTC)
targets = ["carts","catalogue","orders","payment","user"]
start = pd.to_datetime("2024-01-20T14:06:53Z", utc=True)
end   = pd.to_datetime("2024-01-20T14:36:53Z", utc=True)

# 1) Compute global p5,p50,p95 per (cmdb_id,kpi_name) using full series (no time filter)
metric_targets_full = df_metric[df_metric['cmdb_id'].isin(targets)].copy()

metric_global = metric_targets_full.groupby(['cmdb_id','kpi_name'])['value'].agg(
    global_count='count',
    p5=lambda x: x.quantile(0.05),
    p50=lambda x: x.quantile(0.50),
    p95=lambda x: x.quantile(0.95)
).reset_index()

# Keep full precision for computation, but also store rounded for display
metric_global[['p5_r','p50_r','p95_r']] = metric_global[['p5','p50','p95']].round(4)

# 2) Filter metric.csv to the incident window
metric_in = metric_targets_full[(metric_targets_full['timestamp'] >= start) & (metric_targets_full['timestamp'] <= end)].copy()

# 3) Merge global thresholds into in-window rows and determine anomalies
if metric_in.shape[0] > 0:
    metric_in = metric_in.merge(metric_global, on=['cmdb_id','kpi_name'], how='left')
    # Exclude KPI-series where p5==p95==0 from counting as anomalies
    metric_in['exclude_zero_const'] = (metric_in['p5'] == 0.0) & (metric_in['p95'] == 0.0)
    # Anomaly flag using equality as requested, but false for excluded series
    metric_in['is_anom'] = ((metric_in['value'] <= metric_in['p5']) | (metric_in['value'] >= metric_in['p95'])) & (~metric_in['exclude_zero_const'])
    # Determine threshold used per anomaly row: p95 if value >= p95 else p5
    metric_in['used_threshold'] = np.where(metric_in['value'] >= metric_in['p95'], metric_in['p95'], metric_in['p5'])
    # 4) Compute severity = abs(value - threshold)/max(abs(threshold),1e-9)
    metric_in['severity'] = np.abs(metric_in['value'] - metric_in['used_threshold']) / np.maximum(np.abs(metric_in['used_threshold']), 1e-9)
    # For non-anomaly rows severity is 0
    metric_in.loc[~metric_in['is_anom'], 'severity'] = 0.0
else:
    # empty DataFrame with expected columns
    metric_in = pd.DataFrame(columns=list(metric_targets_full.columns) + ['global_count','p5','p50','p95','p5_r','p50_r','p95_r','exclude_zero_const','is_anom','used_threshold','severity'])

# 5) Aggregate per component
# Focus only on anomaly rows for counts and severity
anom_rows = metric_in[metric_in['is_anom']].copy()

if anom_rows.shape[0] > 0:
    # per-component aggregates
    comp_agg = anom_rows.groupby('cmdb_id').agg(
        total_anomaly_events = ('is_anom','sum'),
        num_kpi_series_with_anomaly = ('kpi_name', lambda s: s.nunique()),
        earliest_anomaly_ts = ('timestamp', 'min'),
        max_severity = ('severity','max')
    ).reset_index()
    comp_agg['earliest_anomaly_iso'] = comp_agg['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').replace({'':""})
    comp_agg['max_severity'] = comp_agg['max_severity'].round(3)
    comp_agg = comp_agg.loc[:, ['cmdb_id','total_anomaly_events','num_kpi_series_with_anomaly','earliest_anomaly_iso','max_severity']]
else:
    comp_agg = pd.DataFrame({
        'cmdb_id': targets,
        'total_anomaly_events': [0]*len(targets),
        'num_kpi_series_with_anomaly': [0]*len(targets),
        'earliest_anomaly_iso': [""]*len(targets),
        'max_severity': [0.0]*len(targets)
    })

# Ensure all target components are present in comp_agg (fill zeros if none)
comp_agg = pd.DataFrame({'cmdb_id': targets}).merge(comp_agg, on='cmdb_id', how='left').fillna({
    'total_anomaly_events':0,
    'num_kpi_series_with_anomaly':0,
    'earliest_anomaly_iso':"",
    'max_severity':0.0
})
comp_agg['total_anomaly_events'] = comp_agg['total_anomaly_events'].astype(int)
comp_agg['num_kpi_series_with_anomaly'] = comp_agg['num_kpi_series_with_anomaly'].astype(int)
comp_agg['max_severity'] = comp_agg['max_severity'].round(3)

# 6) For each component, list top 3 KPI series by anomaly count with requested columns
if anom_rows.shape[0] > 0:
    kpi_agg = anom_rows.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count=('is_anom','sum'),
        max_severity=('severity','max'),
        earliest_anomaly_ts = ('timestamp','min')
    ).reset_index()
    kpi_agg['max_severity'] = kpi_agg['max_severity'].round(3)
    kpi_agg['earliest_anomaly_iso'] = kpi_agg['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    # sort and pick top 3 per component
    kpi_agg = kpi_agg.sort_values(['cmdb_id','anomaly_count','max_severity'], ascending=[True, False, False])
    top_kpis = kpi_agg.groupby('cmdb_id').head(3).reset_index(drop=True)
    top_kpis = top_kpis.loc[:, ['cmdb_id','kpi_name','anomaly_count','max_severity','earliest_anomaly_iso']]
else:
    top_kpis = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','max_severity','earliest_anomaly_iso'])

# Final compact outputs:
# - comp_agg sorted by total_anomaly_events desc
# - top_kpis limited rows (up to 15 rows since 5 components * 3)
comp_agg_sorted = comp_agg.sort_values('total_anomaly_events', ascending=False).reset_index(drop=True)
top_kpis_limited = top_kpis.sort_values(['cmdb_id','anomaly_count'], ascending=[True, False]).reset_index(drop=True)

# Display compact summaries
comp_agg_sorted, top_kpis_limited

```
Out[3]:
```
Summary (components: carts, catalogue, orders, payment, user; incident window 2024-01-20T14:06:53 → 2024-01-20T14:36:53 UTC)

Aggregate per component (sorted by total anomaly events)
- payment: total_anomaly_events=56, num_kpi_series_with_anomaly=7, earliest_anomaly=2024-01-20T14:09:00+00:00, max_severity=0.164
- catalogue: total_anomaly_events=42, num_kpi_series_with_anomaly=6, earliest_anomaly=2024-01-20T14:09:00+00:00, max_severity=0.194
- user: total_anomaly_events=30, num_kpi_series_with_anomaly=6, earliest_anomaly=2024-01-20T14:09:00+00:00, max_severity=0.049
- carts: total_anomaly_events=24, num_kpi_series_with_anomaly=6, earliest_anomaly=2024-01-20T14:09:00+00:00, max_severity=0.705
- orders: total_anomaly_events=24, num_kpi_series_with_anomaly=6, earliest_anomaly=2024-01-20T14:09:00+00:00, max_severity=0.781

Notes on method: global p5/p50/p95 were computed from the full series. In-window samples were flagged as anomalies when value <= p5 or >= p95 (equality included). KPI series with p5==p95==0 were excluded from being counted as anomalies.

Top 3 anomalous KPI series per component (kpi_name, anomaly_count, max_severity, earliest_anomaly_iso)
- carts:
  1) latency-90 — anomaly_count=4, max_severity=0.705, earliest=2024-01-20T14:09:00+00:00
  2) cpu       — anomaly_count=4, max_severity=0.681, earliest=2024-01-20T14:09:00+00:00
  3) latency-50— anomaly_count=4, max_severity=0.215, earliest=2024-01-20T14:09:00+00:00

- catalogue:
  1) socket — anomaly_count=22, max_severity=0.194, earliest=2024-01-20T14:09:00+00:00
  2) mem    — anomaly_count=4,  max_severity=0.071, earliest=2024-01-20T14:09:00+00:00
  3) cpu    — anomaly_count=4,  max_severity=0.048, earliest=2024-01-20T14:09:00+00:00

- orders:
  1) latency-90 — anomaly_count=4, max_severity=0.781, earliest=2024-01-20T14:09:00+00:00
  2) latency-50 — anomaly_count=4, max_severity=0.347, earliest=2024-01-20T14:09:00+00:00
  3) cpu        — anomaly_count=4, max_severity=0.339, earliest=2024-01-20T14:09:00+00:00

- payment:
  1) socket — anomaly_count=22, max_severity=0.000, earliest=2024-01-20T14:09:00+00:00
  2) diskio — anomaly_count=14, max_severity=0.004, earliest=2024-01-20T14:10:00+00:00
  3) mem    — anomaly_count=4,  max_severity=0.164, earliest=2024-01-20T14:09:00+00:00

- user:
  1) socket   — anomaly_count=10, max_severity=0.000, earliest=2024-01-20T14:09:00+00:00
  2) latency-50 — anomaly_count=4, max_severity=0.049, earliest=2024-01-20T14:09:00+00:00
  3) workload   — anomaly_count=4, max_severity=0.026, earliest=2024-01-20T14:09:00+00:00

Interpretation highlights
- payment and catalogue have the largest counts of anomaly events (payment 56, catalogue 42). For both, the socket KPI shows many anomalies (22 events each).
- orders and carts show the largest single-KPI severities (orders latency-90 max severity ≈0.781; carts latency-90 ≈0.705 and cpu ≈0.681), indicating larger deviations relative to thresholds for those KPIs.
- Many earliest anomalies occur at ~2024-01-20T14:09:00+00:00.
- KPI series with p5==p95==0 were explicitly excluded from anomaly counts; where those series existed, they were not counted here.

If you want, I can:
- show example anomalous in-window rows for a specific component/KPI, or
- re-evaluate using a different anomaly rule (e.g., strict > p95 / < p5 or exclude small severities).

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  total_anomaly_events  num_kpi_series_with_anomaly      earliest_anomaly_iso  max_severity
0    payment                    56                            7  2024-01-20T14:09:00+0000         0.164
1  catalogue                    42                            6  2024-01-20T14:09:00+0000         0.194
2       user                    30                            6  2024-01-20T14:09:00+0000         0.049
3      carts                    24                            6  2024-01-20T14:09:00+0000         0.705
4     orders                    24                            6  2024-01-20T14:09:00+0000         0.781,       cmdb_id    kpi_name  anomaly_count  max_severity      earliest_anomaly_iso
0       carts  latency-90              4         0.705  2024-01-20T14:09:00+0000
1       carts         cpu              4         0.681  2024-01-20T14:09:00+0000
2       carts  latency-50              4         0.215  2024-01-20T14:09:00+0000
3   catalogue      socket             22         0.194  2024-01-20T14:09:00+0000
4   catalogue         mem              4         0.071  2024-01-20T14:09:00+0000
5   catalogue         cpu              4         0.048  2024-01-20T14:09:00+0000
6      orders  latency-90              4         0.781  2024-01-20T14:09:00+0000
7      orders  latency-50              4         0.347  2024-01-20T14:09:00+0000
8      orders         cpu              4         0.339  2024-01-20T14:09:00+0000
9     payment      socket             22         0.000  2024-01-20T14:09:00+0000
10    payment      diskio             14         0.004  2024-01-20T14:10:00+0000
11    payment         mem              4         0.164  2024-01-20T14:09:00+0000
12       user      socket             10         0.000  2024-01-20T14:09:00+0000
13       user  latency-50              4         0.049  2024-01-20T14:09:00+0000
14       user    workload              4         0.026  2024-01-20T14:09:00+0000)```
```